# Laboratorio (v2): ¿cuánto cambian recall y precisión al repartir por tiempo en lugar de al azar?

**Pregunta:** ¿cuánto cambian el recall y la precisión de una regresión logística en detección de intrusiones con reparto por tiempo en lugar de al azar?

**Hipótesis (fijada antes de medir):** ambas bajan con reparto por tiempo.

**Diseño (comparación pareada):**
- Mismo dataset, mismas variables, mismo modelo (regresión logística), mismo umbral (0,5).
- Lo único que cambia es **cómo se reparte** entre entrenamiento y prueba.
- 5 "pliegues" temporales de origen móvil (se entrena con todo el pasado y se prueba con el bloque siguiente). Para cada pliegue, el reparto al azar usa **los mismos datos y los mismos tamaños**, repetido varias veces.

**Cambios de la v2 respecto a la primera ejecución:**
1. **Métricas sin definir:** si un bloque de prueba no tiene ataques, el recall no existe (0/0). Ya no se fuerza a 0: se marca el pliegue como *no válido* y se excluye del veredicto.
2. **Tasa de falsos positivos (FPR)** como métrica añadida: de las conexiones normales, cuántas disparan alarma. La precisión depende de la proporción de ataques de cada bloque; la FPR no. *Esta métrica se añadió **después** de ver la primera ejecución, así que es exploratoria y no forma parte de la hipótesis original.*
3. **Bloques por tiempo real y sin mezclar capturas:** el dataset son dos capturas (A y B) separadas por un hueco de semanas. Cada captura se divide en bloques de igual duración y ningún bloque cruza el hueco. Además hay un **caso duro**: entrenar con toda la captura A y probar con toda la captura B.

**Después de medir, el laboratorio busca el porqué:** gemelas, tipos de ataque nuevos, atajo de la IP y si el tráfico normal cambia entre capturas.

## 0. Configuración (decide esto ANTES de ejecutar)

In [ ]:
import time, warnings, math
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import NearestNeighbors

warnings.filterwarnings("ignore")
pd.set_option("display.width", 170)
pd.set_option("display.max_columns", 30)

# ---- Parámetros del experimento ------------------------------------------
DATA_DIR = Path("UNSW-NB15")   # carpeta con UNSW-NB15_1.csv ... UNSW-NB15_4.csv
SAMPLE_FRAC = 1.0              # 1.0 = todos los datos (lento, ~30-40 min). 0.25 para probar rápido.
BLOCKS_PER_CAPTURE = 3         # cada captura (A y B) se divide en 3 bloques de igual duración -> 6 bloques, 5 pliegues
N_RANDOM_REPEATS = 10          # repeticiones del reparto al azar por pliegue
N_RANDOM_REPEATS_HARD = 5      # repeticiones del reparto al azar en el caso duro (A -> B)
THRESHOLD = 0.5                # umbral fijo (estudiarlo es tarea del capítulo siguiente)

# ---- Reglas de decisión (fíjalas antes de mirar resultados) ----------------
BIG_DIFF = 0.05          # "diferencia grande" en recall/precisión = 5 puntos porcentuales
MIN_FRAC_AGREE = 0.75    # fracción de pliegues VÁLIDOS en los que debe cumplirse la regla
MIN_TEST_ATTACKS = 100   # un pliegue solo es válido si prueba y entrenamiento tienen al menos tantos ataques
FPR_RATIO_BIG = 2.0      # EXPLORATORIA (añadida tras ver la 1ª ejecución): FPR "grande" = se multiplica por 2
SEED = 42

## 1. Carga y limpieza

Dataset: **UNSW-NB15** (los 4 CSV "en bruto"). Tiene `Stime`/`Ltime` como marcas de tiempo Unix y trae IP de origen y destino.
Los CSV no traen cabecera; los nombres de columna van en el código.

In [ ]:
COLS = ["srcip","sport","dstip","dsport","proto","state","dur","sbytes","dbytes","sttl","dttl",
        "sloss","dloss","service","Sload","Dload","Spkts","Dpkts","swin","dwin","stcpb","dtcpb",
        "smeansz","dmeansz","trans_depth","res_bdy_len","Sjit","Djit","Stime","Ltime","Sintpkt",
        "Dintpkt","tcprtt","synack","ackdat","is_sm_ips_ports","ct_state_ttl","ct_flw_http_mthd",
        "is_ftp_login","ct_ftp_cmd","ct_srv_src","ct_srv_dst","ct_dst_ltm","ct_src_ltm",
        "ct_src_dport_ltm","ct_dst_sport_ltm","ct_dst_src_ltm","attack_cat","Label"]
assert len(COLS) == 49

ID_COLS    = ["srcip", "sport", "dstip", "dsport"]   # identificadores: fuera del modelo base
TIME_COLS  = ["Stime", "Ltime"]                      # el tiempo sirve para repartir, NO como variable
LABEL_COLS = ["Label", "attack_cat"]
CAT_COLS   = ["proto", "state", "service"]
NUM_COLS   = [c for c in COLS if c not in ID_COLS + TIME_COLS + LABEL_COLS + CAT_COLS]
FEAT_COLS  = NUM_COLS + CAT_COLS


def load_unsw(data_dir):
    files = sorted(Path(data_dir).glob("UNSW-NB15_[1-4].csv"))
    assert len(files) == 4, f"Esperaba 4 ficheros UNSW-NB15_[1-4].csv en {data_dir}, encontré {len(files)}"
    return pd.concat([pd.read_csv(f, header=None, names=COLS, low_memory=False) for f in files],
                     ignore_index=True)


df = load_unsw(DATA_DIR)
print("Filas cargadas:", len(df))

# --- limpieza ---
for c in ["sport", "dsport", "ct_ftp_cmd", "ct_flw_http_mthd", "is_ftp_login", "Stime", "Ltime"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")
for c in CAT_COLS + ["srcip", "dstip"]:
    df[c] = df[c].astype(str).str.strip()
df["attack_cat"] = df["attack_cat"].fillna("Normal").astype(str).str.strip().replace({"": "Normal", "Backdoors": "Backdoor"})
df["Label"] = pd.to_numeric(df["Label"], errors="coerce").astype(int)
df = df.dropna(subset=["Stime"])

# --- orden temporal (la base de todo el experimento) ---
df = df.sort_values("Stime", kind="mergesort").reset_index(drop=True)

if SAMPLE_FRAC < 1.0:   # submuestreo uniforme: conserva el orden temporal
    df = df.sample(frac=SAMPLE_FRAC, random_state=SEED).sort_values("Stime", kind="mergesort").reset_index(drop=True)

y = df["Label"].astype(int)
n = len(df)
print(f"Filas usadas: {n:,} | ataques: {y.sum():,} ({y.mean():.1%})")

### 1.1 Las dos capturas y los bloques temporales
Antes de fiarnos del reparto por tiempo hay que ver cómo es el tiempo: el dataset son **dos capturas separadas por un hueco enorme**. Se detecta el hueco, se etiqueta cada fila como captura A o B y se divide cada captura en bloques de **igual duración** (ninguno cruza el hueco).

In [ ]:
t = pd.to_datetime(df["Stime"], unit="s")
st = df["Stime"].values
gaps = np.diff(st)
assert gaps.max() > 24 * 3600, "No detecto dos capturas separadas por un hueco grande"
split_pos = int(np.argmax(gaps)) + 1                 # primera fila de la captura B
capture = np.zeros(n, dtype=int); capture[split_pos:] = 1

print("Stime ordenado:", df["Stime"].is_monotonic_increasing)
print(f"Hueco entre capturas: {gaps.max() / 3600:.0f} horas")

caps = []
for c, name in [(0, "A"), (1, "B")]:
    idx = np.where(capture == c)[0]
    caps.append(dict(captura=name, desde=t.iloc[idx[0]], hasta=t.iloc[idx[-1]],
                     horas=(st[idx[-1]] - st[idx[0]]) / 3600, filas=len(idx),
                     ataques=int(y.iloc[idx].sum()), pct_ataques=y.iloc[idx].mean() * 100))
display(pd.DataFrame(caps).round(2))


def time_edges(lo, hi, k):
    # divide las filas [lo, hi) en k bloques de igual DURACIÓN (no igual nº de filas)
    ts = np.linspace(st[lo], st[hi - 1], k + 1)
    inner = lo + np.searchsorted(st[lo:hi], ts[1:-1], side="left")
    return [lo] + inner.tolist() + [hi]


K = BLOCKS_PER_CAPTURE
edges = np.array(time_edges(0, split_pos, K) + time_edges(split_pos, n, K)[1:])
N_BLOCKS = 2 * K
blocks = pd.DataFrame({
    "bloque": range(N_BLOCKS),
    "captura": ["A"] * K + ["B"] * K,
    "desde": [t.iloc[edges[i]] for i in range(N_BLOCKS)],
    "hasta": [t.iloc[edges[i + 1] - 1] for i in range(N_BLOCKS)],
    "horas": [(st[edges[i + 1] - 1] - st[edges[i]]) / 3600 for i in range(N_BLOCKS)],
    "filas": np.diff(edges),
    "ataques": [int(y.iloc[edges[i]:edges[i + 1]].sum()) for i in range(N_BLOCKS)],
    "% ataques": [y.iloc[edges[i]:edges[i + 1]].mean() * 100 for i in range(N_BLOCKS)],
}).round(2)
blocks

## 2. Piezas del experimento
Modelo, métricas (con "sin definir" cuando la división es 0/0) y los dos tipos de reparto, con el mismo tamaño de entrenamiento y de prueba.

In [ ]:
def log1p_pos(a):
    return np.log1p(np.clip(a, 0, None))


def make_model(num_cols, cat_cols, min_freq=100):
    # Regresión logística con preprocesado mínimo (log + estandarizar + one-hot).
    parts = []
    if num_cols:
        parts.append(("num", Pipeline([("imp", SimpleImputer(strategy="median")),
                                       ("log", FunctionTransformer(log1p_pos, feature_names_out="one-to-one")),
                                       ("sc", StandardScaler())]), list(num_cols)))
    if cat_cols:
        parts.append(("cat", OneHotEncoder(handle_unknown="infrequent_if_exist",
                                           min_frequency=min_freq), list(cat_cols)))
    return Pipeline([("prep", ColumnTransformer(parts)),
                     ("clf", LogisticRegression(max_iter=1000))])


def metrics(y_true, proba, thr=THRESHOLD):
    # Calculado a mano: si una división es 0/0 el resultado es NaN ("sin definir"), nunca 0.
    y_true = np.asarray(y_true).astype(int)
    pred = (np.asarray(proba) >= thr).astype(int)
    tp = int(((pred == 1) & (y_true == 1)).sum()); fp = int(((pred == 1) & (y_true == 0)).sum())
    fn = int(((pred == 0) & (y_true == 1)).sum()); tn = int(((pred == 0) & (y_true == 0)).sum())
    P, Nn, tot = tp + fn, fp + tn, len(y_true)
    div = lambda a, b: a / b if b > 0 else np.nan
    return dict(
        accuracy=(tp + tn) / tot,
        precision=div(tp, tp + fp) if P > 0 else np.nan,  # de mis alertas, cuántas eran ataques (sin ataques en la prueba: sin definir)
        recall=div(tp, P),                               # de los ataques reales, cuántos capturé
        f1=div(2 * tp, 2 * tp + fp + fn) if P > 0 else np.nan,
        fpr=div(fp, Nn),                                 # de las conexiones normales, cuántas disparan alarma
        pr_auc=average_precision_score(y_true, proba) if P > 0 else np.nan,
        prevalence=P / tot,
        acc_todo_normal=1 - P / tot,                     # accuracy del modelo tonto "todo normal"
        n_test_pos=P,
    )


METRIC_KEYS = list(metrics(np.array([0, 1]), np.array([0.2, 0.8])).keys())


def temporal_split(train_end, test_end):
    return np.arange(0, train_end), np.arange(train_end, test_end)


def random_split(train_end, test_end, seed):
    # Mismas filas (las primeras test_end) y mismos tamaños que el reparto temporal, pero mezclando al azar.
    pool = np.random.default_rng(seed).permutation(test_end)
    return pool[:train_end], pool[train_end:test_end]


def fit_predict(cols_num, cols_cat, tr, te):
    m = make_model(cols_num, cols_cat)
    cols = list(cols_num) + list(cols_cat)
    m.fit(df.iloc[tr][cols], y.iloc[tr])
    return m.predict_proba(df.iloc[te][cols])[:, 1]


def run_one(tr, te, return_proba=False):
    if y.iloc[tr].nunique() < 2:    # entrenamiento sin ataques (o sin normales): no hay modelo posible
        out = {k: np.nan for k in METRIC_KEYS}; out["n_test_pos"] = int(y.iloc[te].sum()); p = None
    else:
        p = fit_predict(NUM_COLS, CAT_COLS, tr, te)
        out = metrics(y.iloc[te].values, p)
    return (out, p) if return_proba else out


def recall_by_cat(te, proba):
    pred = (proba >= THRESHOLD).astype(int)
    d = pd.DataFrame({"cat": df["attack_cat"].iloc[te].values, "y": y.iloc[te].values, "pred": pred})
    return d[d.y == 1].groupby("cat")["pred"].mean()

## 3. Experimento principal: azar vs tiempo en 5 pliegues
Para cada pliegue `k`: entrenar con los bloques `0..k` y probar con el bloque `k+1`. Los bloques 0-2 son la captura A y los 3-5 la captura B, así que **el pliegue 2 prueba con el primer bloque de la captura B** (el primer salto real entre capturas).
El reparto al azar usa las mismas filas y los mismos tamaños, repetido `N_RANDOM_REPEATS` veces.

In [ ]:
rows = []
t0 = time.time()
for k in range(N_BLOCKS - 1):
    train_end, test_end = edges[k + 1], edges[k + 2]
    base = dict(fold=k, n_train=train_end, n_test=test_end - train_end)

    # --- por tiempo ---
    tr, te = temporal_split(train_end, test_end)
    rows.append(dict(**base, split="tiempo", rep=0, n_train_pos=int(y.iloc[tr].sum()), **run_one(tr, te)))

    # --- al azar (repetido) ---
    for r in range(N_RANDOM_REPEATS):
        tr, te = random_split(train_end, test_end, seed=SEED + 1000 * k + r)
        rows.append(dict(**base, split="azar", rep=r, n_train_pos=int(y.iloc[tr].sum()), **run_one(tr, te)))
    print(f"Pliegue {k} listo ({time.time() - t0:.0f} s)")

res = pd.DataFrame(rows)

# --- ¿qué pliegues son válidos? ---
fold_info = res[res.split == "tiempo"].set_index("fold")[["n_train", "n_train_pos", "n_test", "n_test_pos"]].copy()
fold_info["captura de prueba"] = [blocks.loc[k + 1, "captura"] for k in fold_info.index]
fold_info["% ataques prueba"] = (fold_info.n_test_pos / fold_info.n_test * 100).round(2)
fold_info["valido"] = (fold_info.n_test_pos >= MIN_TEST_ATTACKS) & (fold_info.n_train_pos >= MIN_TEST_ATTACKS)
valid_folds = list(fold_info.index[fold_info.valido])
print("Pliegues válidos:", valid_folds, "| excluidos:", [k for k in fold_info.index if k not in valid_folds])
fold_info

### 3.1 Tablas por métrica y veredicto sobre la hipótesis
En cada tabla: media y variabilidad del reparto al azar, valor por tiempo y diferencia (tiempo − azar). Un valor negativo significa que el reparto por tiempo da una nota **más baja**. Los pliegues no válidos salen con `valido = False` y **no cuentan** en el veredicto.

In [ ]:
def tabla(metric):
    az = res[res.split == "azar"].groupby("fold")[metric]
    out = pd.DataFrame({
        "azar_media": az.mean(), "azar_std": az.std(), "azar_min": az.min(), "azar_max": az.max(),
        "tiempo": res[res.split == "tiempo"].set_index("fold")[metric],
    })
    out["delta"] = out["tiempo"] - out["azar_media"]
    if metric == "fpr":
        out["ratio"] = out["tiempo"] / out["azar_media"].replace(0, np.nan)
    out["valido"] = fold_info["valido"]
    return out

for m in ["recall", "precision", "fpr", "f1", "pr_auc", "accuracy"]:
    print(f"\n=== {m} ===")
    display(tabla(m).round(4))

In [ ]:
def need_folds(n_valid):
    return max(1, math.ceil(MIN_FRAC_AGREE * n_valid))


def veredicto(metric):
    # Regla para recall/precisión/F1/PR-AUC/accuracy: ¿cae más de BIG_DIFF en la fracción exigida de pliegues válidos?
    tb = tabla(metric).loc[valid_folds]
    d, nv = tb["delta"], len(tb)
    if nv == 0:
        print(f"{metric:>9}: sin pliegues válidos"); return
    need = need_folds(nv)
    n_drop, n_up = int((d < -BIG_DIFF).sum()), int((d > BIG_DIFF).sum())
    fuera = int((tb["tiempo"] < tb["azar_min"]).sum())
    if d.mean() < -BIG_DIFF and n_drop >= need:
        v = "BAJA (hipótesis confirmada)"
    elif d.mean() > BIG_DIFF and n_up >= need:
        v = "SUBE (hipótesis refutada)"
    else:
        v = "SIN CAMBIO CLARO / INCONSISTENTE"
    print(f"{metric:>9}: media Δ = {d.mean():+.3f} | caída > {BIG_DIFF:.2f} en {n_drop}/{nv} (se exigen {need}) "
          f"| tiempo < mínimo del azar en {fuera}/{nv} -> {v}")


def veredicto_fpr():
    # EXPLORATORIA: ¿se multiplica la tasa de falsos positivos por FPR_RATIO_BIG o más?
    tb = tabla("fpr").loc[valid_folds]
    nv = len(tb)
    if nv == 0:
        print("      fpr: sin pliegues válidos"); return
    need = need_folds(nv)
    ratio = tb["ratio"]
    n_up = int((ratio >= FPR_RATIO_BIG).sum()); n_down = int((ratio <= 1 / FPR_RATIO_BIG).sum())
    if n_up >= need:
        v = f"SE MULTIPLICA por tiempo (≥{FPR_RATIO_BIG:.0f}x): más falsas alarmas"
    elif n_down >= need:
        v = "SE REDUCE por tiempo"
    else:
        v = "SIN CAMBIO CLARO / INCONSISTENTE"
    print(f"      fpr: ratio mediano tiempo/azar = {ratio.median():.2f} | ratio ≥ {FPR_RATIO_BIG:.0f} en {n_up}/{nv} (se exigen {need}) -> {v}")


print(f"Pliegues válidos: {valid_folds}\n")
print("VEREDICTO SOBRE LA HIPÓTESIS (recall y precisión)")
for m in ["recall", "precision"]:
    veredicto(m)
print("\n(Secundarias)")
for m in ["f1", "pr_auc", "accuracy"]:
    veredicto(m)
print("\n(Exploratoria, añadida tras ver la 1ª ejecución)")
veredicto_fpr()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, m in zip(axes, ["recall", "precision", "fpr"]):
    first = True
    for k in valid_folds:
        v = res[(res.fold == k) & (res.split == "azar")][m]
        ax.scatter([k] * len(v), v, color="grey", alpha=0.5, s=18, label="azar (repeticiones)" if first else None)
        ax.scatter(k, res[(res.fold == k) & (res.split == "tiempo")][m].iloc[0], color="crimson", s=70,
                   marker="D", label="tiempo" if first else None)
        first = False
    ax.set_title(m + (" (falsos positivos / normales)" if m == "fpr" else "")); ax.set_xlabel("pliegue (solo válidos)")
    ax.set_xticks(valid_folds); ax.grid(alpha=0.3)
axes[0].legend()
plt.tight_layout(); plt.show()

### 3.2 La trampa del accuracy
Accuracy del modelo frente al del modelo tonto «todo normal» (solo pliegues válidos). Si el accuracy apenas supera al tonto, no dice nada.

In [ ]:
res[res.fold.isin(valid_folds)].groupby(["fold", "split"])[["accuracy", "acc_todo_normal", "prevalence", "recall", "fpr"]].mean().round(4)

## 4. Caso duro: entrenar con toda la captura A y probar con toda la captura B
El salto temporal más grande posible en este dataset. El reparto al azar usa **todos** los datos con los mismos tamaños (entrenamiento = nº de filas de A, prueba = nº de filas de B).

In [ ]:
trA, teB = np.arange(0, split_pos), np.arange(split_pos, n)
print(f"Entrenamiento (captura A): {len(trA):,} filas, {int(y.iloc[trA].sum()):,} ataques ({y.iloc[trA].mean():.1%})")
print(f"Prueba        (captura B): {len(teB):,} filas, {int(y.iloc[teB].sum()):,} ataques ({y.iloc[teB].mean():.1%})")

hard_rows, probas = [], {}
mt, pt = run_one(trA, teB, return_proba=True)
hard_rows.append(dict(split="tiempo", rep=0, **mt)); probas["tiempo"] = pt
for r in range(N_RANDOM_REPEATS_HARD):
    perm = np.random.default_rng(SEED + 777 + r).permutation(n)
    tr, te = perm[:len(trA)], perm[len(trA):]
    mr, pr = run_one(tr, te, return_proba=True)
    hard_rows.append(dict(split="azar", rep=r, **mr))
    if r == 0:
        probas["azar"], te_hard_random = pr, te
    print(f"  azar {r + 1}/{N_RANDOM_REPEATS_HARD} listo")
hard = pd.DataFrame(hard_rows)

HK = ["accuracy", "precision", "recall", "f1", "fpr", "pr_auc", "prevalence", "acc_todo_normal"]
hard_tab = pd.DataFrame({
    "azar (media)": hard[hard.split == "azar"][HK].mean(),
    "azar (std)": hard[hard.split == "azar"][HK].std(),
    "tiempo (A->B)": hard[hard.split == "tiempo"][HK].iloc[0],
})
hard_tab["delta"] = hard_tab["tiempo (A->B)"] - hard_tab["azar (media)"]
hard_tab.round(4)

### 4.1 ¿Qué tipos de ataque hay en A y en B?
Un tipo de ataque que aparece en B y no (o casi no) en A es **cambio real** (deriva), no fuga de datos: ningún modelo puede detectar lo que nunca vio.

In [ ]:
cat_tab = pd.DataFrame({
    "n en A (train)": df["attack_cat"].iloc[trA][y.iloc[trA] == 1].value_counts(),
    "n en B (test)": df["attack_cat"].iloc[teB][y.iloc[teB] == 1].value_counts(),
    "recall tiempo (A->B)": recall_by_cat(teB, probas["tiempo"]),
    "recall azar (1ª rep.)": recall_by_cat(te_hard_random, probas["azar"]),
}).fillna(0)
cat_tab[["n en A (train)", "n en B (test)"]] = cat_tab[["n en A (train)", "n en B (test)"]].astype(int)
display(cat_tab.sort_values("n en B (test)", ascending=False).round(3))
nuevos = cat_tab.index[(cat_tab["n en A (train)"] < 50) & (cat_tab["n en B (test)"] >= 50)].tolist()
print("Tipos con ≥50 ejemplos en B y <50 en A (prácticamente nuevos):", nuevos if nuevos else "ninguno")

### 4.2 ¿Cambia el tráfico NORMAL entre capturas?
Si el tráfico normal de B es distinto del de A, el modelo lo trata como sospechoso y suben las falsas alarmas. Se comprueba entrenando un clasificador que intente distinguir A de B **usando solo tráfico normal**: AUC ≈ 0,5 = indistinguibles; AUC cercano a 1 = muy distintos.

In [ ]:
norm_idx = np.where(y.values == 0)[0]
sub = np.random.default_rng(SEED).choice(norm_idx, size=min(len(norm_idx), 300_000), replace=False)
tr_i, te_i = train_test_split(sub, test_size=0.3, stratify=capture[sub], random_state=SEED)
adv = make_model(NUM_COLS, CAT_COLS).fit(df.iloc[tr_i][FEAT_COLS], capture[tr_i])
auc = roc_auc_score(capture[te_i], adv.predict_proba(df.iloc[te_i][FEAT_COLS])[:, 1])
print(f"AUC para distinguir captura A de B solo con tráfico normal: {auc:.3f}")
print("(0.5 = no se distinguen; >0.8 = tráfico normal muy distinto entre capturas)")
try:
    names = adv.named_steps["prep"].get_feature_names_out()
    coefs = pd.Series(adv.named_steps["clf"].coef_[0], index=names)
    print("\nVariables que más separan A de B (coeficiente):")
    display(coefs.reindex(coefs.abs().sort_values(ascending=False).index).head(10).round(2))
except Exception as e:
    print("(No se pudieron obtener los nombres de variables:", e, ")")

## 5. ¿Por qué cambia? Diagnósticos con un reparto 70 % / 30 %
Un único reparto (por tiempo y al azar, mismos tamaños), igual que en la primera ejecución para poder comparar.

In [ ]:
train_end, test_end = int(n * 0.7), n
tr_t, te_t = temporal_split(train_end, test_end)
tr_r, te_r = random_split(train_end, test_end, seed=SEED)

proba_t = fit_predict(NUM_COLS, CAT_COLS, tr_t, te_t)
proba_r = fit_predict(NUM_COLS, CAT_COLS, tr_r, te_r)
m_t, m_r = metrics(y.iloc[te_t].values, proba_t), metrics(y.iloc[te_r].values, proba_r)
pd.DataFrame({"azar": m_r, "tiempo": m_t}).round(4)

### 5.1 Gemelas exactas
¿Qué porcentaje de las filas de prueba tiene una fila **idéntica** (en todas las variables del modelo) en entrenamiento?

In [ ]:
h = pd.util.hash_pandas_object(df[FEAT_COLS], index=False).values

def twin_share(tr, te):
    train_hashes = set(h[tr])
    return np.array([x in train_hashes for x in h[te]])

tw_t, tw_r = twin_share(tr_t, te_t), twin_share(tr_r, te_r)
yt, yr = y.iloc[te_t].values, y.iloc[te_r].values

pd.DataFrame({
    "azar":   {"% filas de prueba con gemela exacta": tw_r.mean() * 100,
               "% de los ATAQUES con gemela exacta": tw_r[yr == 1].mean() * 100},
    "tiempo": {"% filas de prueba con gemela exacta": tw_t.mean() * 100,
               "% de los ATAQUES con gemela exacta": tw_t[yt == 1].mean() * 100},
}).round(1)

### 5.2 Gemelas aproximadas: distancia al vecino más cercano
Distancia (variables numéricas estandarizadas) de cada fila de prueba a su fila de entrenamiento más parecida. Si en el reparto al azar las distancias son mucho más pequeñas, el modelo se examina con conocidas.

In [ ]:
def nn_distances(tr, te, max_train=100_000, max_test=5_000, seed=SEED):
    rng = np.random.default_rng(seed)
    tr_s = rng.choice(tr, size=min(max_train, len(tr)), replace=False)
    te_s = rng.choice(te, size=min(max_test, len(te)), replace=False)
    pre = Pipeline([("imp", SimpleImputer(strategy="median")),
                    ("log", FunctionTransformer(log1p_pos)),
                    ("sc", StandardScaler())]).fit(df.iloc[tr][NUM_COLS])
    A, B = pre.transform(df.iloc[tr_s][NUM_COLS]), pre.transform(df.iloc[te_s][NUM_COLS])
    nn = NearestNeighbors(n_neighbors=1, algorithm="brute", n_jobs=-1).fit(A)
    d, _ = nn.kneighbors(B)
    return d.ravel(), y.iloc[te_s].values

d_r, ys_r = nn_distances(tr_r, te_r)
d_t, ys_t = nn_distances(tr_t, te_t)

print("Distancia mediana al vecino más cercano de entrenamiento")
print(f"  azar  : todas={np.median(d_r):.3f} | ataques={np.median(d_r[ys_r == 1]):.3f}")
print(f"  tiempo: todas={np.median(d_t):.3f} | ataques={np.median(d_t[ys_t == 1]):.3f}")

plt.figure(figsize=(7, 4))
bins = np.linspace(0, np.percentile(np.r_[d_r, d_t], 99), 60)
plt.hist(d_r, bins=bins, alpha=0.6, label="azar", density=True)
plt.hist(d_t, bins=bins, alpha=0.6, label="tiempo", density=True)
plt.xlabel("distancia al vecino más cercano en entrenamiento"); plt.legend(); plt.grid(alpha=0.3); plt.show()

### 5.3 ¿Cuánto de la "nota inflada" son gemelas exactas?
Se evalúa el modelo del reparto al azar **solo con las filas de prueba SIN gemela exacta**. Ojo: quitar gemelas también cambia la proporción de ataques (quedan los casos raros), así que esta tabla indica cuánto pesan las gemelas, no es una "nota real".

In [ ]:
mask = ~tw_r
res_nodup = metrics(yr[mask], proba_r[mask])
pd.DataFrame({
    "azar (todo)": m_r,
    "azar (sin gemelas exactas)": res_nodup,
    "tiempo": m_t,
}).loc[["precision", "recall", "f1", "fpr", "pr_auc", "prevalence"]].round(4)

### 5.4 ¿Qué ataques se pierden por tiempo? (reparto 70/30)
Recall por tipo de ataque en cada reparto, y cuántos ejemplos de cada tipo vio el modelo en entrenamiento.

In [ ]:
cat_all = df["attack_cat"]
out = pd.DataFrame({
    "recall azar": recall_by_cat(te_r, proba_r),
    "recall tiempo": recall_by_cat(te_t, proba_t),
    "n en train (tiempo)": cat_all.iloc[tr_t][y.iloc[tr_t] == 1].value_counts(),
    "n en test (tiempo)": cat_all.iloc[te_t][y.iloc[te_t] == 1].value_counts(),
}).fillna(0)
out["n en train (tiempo)"] = out["n en train (tiempo)"].astype(int)
out["n en test (tiempo)"] = out["n en test (tiempo)"].astype(int)
out.sort_values("n en test (tiempo)", ascending=False).round(3)

### 5.5 ¿Aprende atajos? Prueba con la IP
Tres modelos, en ambos repartos:
- **base**: solo comportamiento de la conexión.
- **base + IP**: añade IP de origen y de destino.
- **solo IP**: no ve nada del tráfico, solo las IP.

In [ ]:
IP_COLS = ["srcip", "dstip"]
variantes = {
    "base":        (NUM_COLS, CAT_COLS),
    "base + IP":   (NUM_COLS, CAT_COLS + IP_COLS),
    "solo IP":     ([], IP_COLS),
}
rows = []
for nombre, (nc, cc) in variantes.items():
    for split_name, (tr, te) in {"azar": (tr_r, te_r), "tiempo": (tr_t, te_t)}.items():
        p = fit_predict(nc, cc, tr, te)
        mm = metrics(y.iloc[te].values, p)
        rows.append(dict(modelo=nombre, reparto=split_name, precision=mm["precision"],
                         recall=mm["recall"], f1=mm["f1"], fpr=mm["fpr"]))
pd.DataFrame(rows).pivot(index="modelo", columns="reparto", values=["precision", "recall", "f1", "fpr"]).round(4)

In [ ]:
atk = df[df.Label == 1]
print("IP de origen distintas en ataques :", atk.srcip.nunique(), "| en tráfico normal:", df[df.Label == 0].srcip.nunique())
print("Top IP de origen en ataques (% de ataques):")
print((atk.srcip.value_counts(normalize=True).head(5) * 100).round(1))
print("\nIP de origen de ataque que también aparecen en tráfico normal:",
      len(set(atk.srcip) & set(df[df.Label == 0].srcip)), "de", atk.srcip.nunique())

## 6. Conclusiones (rellena tú)

Usa los números de las tablas de arriba. Guía:

1. **Hipótesis:** ¿recall y precisión bajan con reparto por tiempo? ¿Más que `BIG_DIFF`? ¿En qué fracción de los pliegues **válidos**? (3.1)
2. **Cuánto:** diferencia media en recall y precisión frente a la variabilidad del reparto al azar (`azar_std` en las tablas).
3. **Falsas alarmas (exploratoria):** ¿cambia la FPR? Recuerda que esta métrica se añadió tras ver la primera ejecución. (3.1)
4. **Caso duro A→B:** ¿cuánto cae todo cuando el salto temporal es máximo? ¿Hay tipos de ataque nuevos? ¿Cambia el tráfico normal? (4, 4.1, 4.2)
5. **Por qué (fuga de datos):** % de gemelas exactas en azar vs tiempo, distancias y qué pasa al quitarlas. (5.1–5.3)
6. **Atajos:** ¿qué pasa con "solo IP" al pasar de azar a tiempo? (5.5)
7. **Accuracy:** ¿cuánto supera al modelo tonto «todo normal»? (3.2)

**Limitaciones a nombrar en el post:** un solo dataset, un solo modelo (lineal, que no memoriza), un solo umbral (0,5), solo dos capturas, pliegues descartados por no tener ataques suficientes, y la FPR como métrica añadida a posteriori.